In [2]:
import os
import gc
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
import rasterio.features
import geopandas as gpd

from shapely.geometry import shape
from datetime import datetime
from pytz import timezone, utc
import matplotlib.patches as patches


# ============================================================
# Optional memory tracking
# ============================================================

try:
    import psutil
    PSUTIL_AVAILABLE = True
except ImportError:
    PSUTIL_AVAILABLE = False
    print("[!] psutil is not installed. Memory tracking will be skipped.")
    print("    To install it, run:")
    print("    python3 -m pip install psutil")


def print_memory(note=""):
    """
    Print current memory usage for this Python process.
    Only works if psutil is installed.
    """
    if not PSUTIL_AVAILABLE:
        return

    process = psutil.Process(os.getpid())
    mem_gb = process.memory_info().rss / 1024**3
    print(f"  memory {note}: {mem_gb:.2f} GB")


# ============================================================
# Parameters
# ============================================================

# Preprocessed SAR GeoTIFFs
original_root = "/Volumes/External/TJ/020_preprocessing/"

# AutoSAR Contrast Ratio output GeoTIFFs
mask_root = '/Volumes/External/TJ/040_segoutput'

# Output directories
output_plot_dir = "/Volumes/External/TJ/050_viz/autoSAR_seg_output"
output_vector_dir = "/Volumes/External/TJ/015_shapefiles/autoSAR_seg_output"

# Meteorological CSV
met_csv_path = "/Volumes/External/TJ_estuary/analysis/TJRTLMET.csv"

# Point-data shapefile
point_shapefile_path = "/Volumes/External/TJ/015_shapefiles/outflow_PB_TJ/Outflow.shp"

# Only keep mask values greater than this threshold
threshold = 0

# Minimum polygon area to keep.
# This is in the raster CRS units.
# If your CRS is meters, this is square meters.
# Increase this if vectorizing is still too slow.
min_polygon_area = 0

# Set this to True if you want to make plots.
# If you only want vector files, change this to False.
make_plots = True

# Set this to True if you want vector GeoPackages.
# If you only want plots, change this to False.
make_vectors = True

# Timezone
pacific = timezone("US/Pacific")


# ============================================================
# Load meteorological data
# ============================================================

print("Loading meteorological data...")

met_df = pd.read_csv(met_csv_path)
met_df.columns = met_df.columns.str.strip()

met_df["DateTimeStamp"] = pd.to_datetime(
    met_df["DateTimeStamp"],
    format="%m/%d/%y %H:%M",
    errors="coerce"
)

# Your met data timestamps are already in Pacific Time
met_df["DateTimeStamp"] = met_df["DateTimeStamp"].dt.tz_localize(
    pacific,
    ambiguous="NaT",
    nonexistent="NaT"
)

met_df = met_df.dropna(subset=["DateTimeStamp"]).sort_values("DateTimeStamp")

print(f"Loaded {len(met_df):,} valid meteorological records.")


# ============================================================
# Load point data once
# ============================================================

print("Loading outflow point shapefile...")

points_gdf_orig = gpd.read_file(point_shapefile_path)

print(f"Loaded {len(points_gdf_orig):,} outflow point features.")


# ============================================================
# Utility functions
# ============================================================

def extract_datetime_from_filename(filename):
    """
    Extract UTC datetime from Sentinel-1 filename.

    Example filename piece:
    20240818T135318

    Returns:
    timezone-aware datetime in UTC, or None if not found.
    """
    parts = filename.split("_")

    for part in parts:
        if part.startswith("20") and "T" in part:
            try:
                return utc.localize(datetime.strptime(part, "%Y%m%dT%H%M%S"))
            except ValueError:
                continue

    return None


def get_nearest_met_data(local_time):
    """
    Find the closest meteorological record to the given local image time.

    Returns:
    wind speed, wind direction, and met timestamp.
    """
    diffs = (met_df["DateTimeStamp"] - local_time).abs()
    idx = diffs.idxmin()
    row = met_df.loc[idx]

    return row["WSpd"], row["Wdir"], row["DateTimeStamp"]


def match_original_path(mask_filename):
    """
    Convert AutoSAR mask filename to the matching preprocessed image path.

    Example:
    S1A_..._pre_JPL1.3_VVDR_cumulative.tif

    becomes:
    /Volumes/External/TJ/020_preprocessing/S1A_..._pre.tif
    """
    original_filename = mask_filename.replace(
        "_JPL1.3_VVDR_cumulative_mask.tif",
        ".tif"
    )

    return os.path.join(original_root, original_filename)


def get_clean_base_name(mask_filename):
    """
    Convert AutoSAR mask filename to a clean base name for outputs.

    Example:
    S1A_..._pre_JPL1.3_VVDR_cumulative.tif

    becomes:
    S1A_..._pre
    """
    base = os.path.basename(mask_filename)

    base = base.replace(
        "_JPL1.3_VVDR_cumulative_mask.tif",
        ""
    )

    return base


def cleanup_loop_variables():
    """
    Force cleanup between files.
    """
    plt.close("all")
    gc.collect()


# ============================================================
# Main processing loop
# ============================================================

os.makedirs(output_plot_dir, exist_ok=True)
os.makedirs(output_vector_dir, exist_ok=True)

processed_count = 0
skipped_count = 0
failed_count = 0

print("\nStarting processing...")
print_memory("before loop")

for root, _, files in os.walk(mask_root):
    for file in files:

        # Skip hidden files and non-mask files
        if file.startswith(".") or not file.endswith("_VVDR_cumulative_mask.tif"):
            continue

        processed_count += 1

        mask_path = os.path.join(root, file)
        original_path = match_original_path(file)
        base_name = get_clean_base_name(file)

        png_name = f"{base_name}_overlay.png"
        out_png = os.path.join(output_plot_dir, png_name)

        gpkg_name = f"{base_name}_mask.gpkg"
        out_gpkg = os.path.join(output_vector_dir, gpkg_name)

        print("\n--------------------------------------------------")
        print(f"File #{processed_count}")
        print(f"Processing: {file}")
        print_memory("at start of file")

        # Skip files already completed
        plot_done = os.path.exists(out_png)
        vector_done = os.path.exists(out_gpkg)

        if make_plots and make_vectors:
            if plot_done and vector_done:
                print(f"  -> plot and vector already exist, skipping: {base_name}")
                skipped_count += 1
                cleanup_loop_variables()
                print_memory("after skip cleanup")
                continue

        elif make_plots and not make_vectors:
            if plot_done:
                print(f"  -> plot already exists, skipping: {base_name}")
                skipped_count += 1
                cleanup_loop_variables()
                print_memory("after skip cleanup")
                continue

        elif make_vectors and not make_plots:
            if vector_done:
                print(f"  -> vector already exists, skipping: {base_name}")
                skipped_count += 1
                cleanup_loop_variables()
                print_memory("after skip cleanup")
                continue

        if not os.path.exists(original_path):
            print(f"[!] Original image not found for: {file}")
            print(f"    Expected path: {original_path}")
            failed_count += 1
            cleanup_loop_variables()
            continue

        try:
            # ====================================================
            # Read mask raster
            # ====================================================

            print("  reading mask raster...")

            with rasterio.open(mask_path) as src_mask:
                mask_arr = src_mask.read(1)
                transform = src_mask.transform
                crs = src_mask.crs
                nodata = src_mask.nodata
                mask_shape = src_mask.shape

            print_memory("after reading mask")

            # Build boolean mask.
            # True where pixels are greater than threshold and not nodata.
            valid_mask = mask_arr > threshold

            if nodata is not None:
                valid_mask &= (mask_arr != nodata)

            # Also remove NaN and infinite values if present
            valid_mask &= np.isfinite(mask_arr)

            n_valid = np.count_nonzero(valid_mask)
            total_pixels = valid_mask.size
            percent_valid = 100 * n_valid / total_pixels

            print(
                f"  valid mask pixels: {n_valid:,} / "
                f"{total_pixels:,} ({percent_valid:.4f}%)"
            )

            # ====================================================
            # Vectorize mask to polygons
            # ====================================================

            if make_vectors and not vector_done:

                if n_valid == 0:
                    print(f"  -> no valid mask pixels for: {file}, skipping vector output.")

                else:
                    print("  creating binary mask for vectorizing...")

                    # Memory-safe binary mask.
                    # Avoids casting NaN values from the original mask array.
                    mask_binary = np.zeros(mask_arr.shape, dtype=np.uint8)
                    mask_binary[valid_mask] = 1

                    print_memory("after creating binary mask")
                    print("  vectorizing mask...")
                    print_memory("before vectorizing")

                    shapes = []
                    raw_polygon_count = 0
                    kept_polygon_count = 0
                    skipped_small_count = 0
                    skipped_empty_count = 0

                    for geom, val in rasterio.features.shapes(
                        mask_binary,
                        mask=valid_mask,
                        transform=transform
                    ):
                        raw_polygon_count += 1

                        polygon = shape(geom)

                        if polygon.is_empty:
                            skipped_empty_count += 1
                            continue

                        if polygon.area < min_polygon_area:
                            skipped_small_count += 1
                            continue

                        shapes.append({
                            "geometry": polygon,
                            "value": int(val)
                        })

                        kept_polygon_count += 1

                        # Progress print every 10,000 raw polygons
                        if raw_polygon_count % 10000 == 0:
                            print(
                                f"    raw polygons: {raw_polygon_count:,}, "
                                f"kept: {kept_polygon_count:,}, "
                                f"small skipped: {skipped_small_count:,}"
                            )
                            print_memory("during vectorizing")

                    print("  vectorizing complete.")
                    print(f"  raw polygons created: {raw_polygon_count:,}")
                    print(f"  kept polygons: {kept_polygon_count:,}")
                    print(f"  skipped small polygons: {skipped_small_count:,}")
                    print(f"  skipped empty polygons: {skipped_empty_count:,}")
                    print_memory("after vectorizing")

                    # Delete binary mask immediately
                    del mask_binary
                    gc.collect()
                    print_memory("after deleting binary mask")

                    if shapes:
                        print("  creating GeoDataFrame...")

                        gdf = gpd.GeoDataFrame(shapes, crs=crs)

                        print_memory("after creating GeoDataFrame")

                        print(f"  writing GeoPackage: {out_gpkg}")

                        gdf.to_file(out_gpkg, driver="GPKG")

                        print(f"  -> wrote GeoPackage: {out_gpkg}")
                        print_memory("after writing GeoPackage")

                        # Delete large vector objects immediately
                        del gdf
                        del shapes
                        gc.collect()
                        print_memory("after deleting vector objects")

                    else:
                        print(f"  -> no polygons left after filtering for: {file}")

            elif make_vectors and vector_done:
                print(f"  -> vector already exists: {out_gpkg}")

            # ====================================================
            # Plotting
            # ====================================================

            if make_plots and not plot_done:

                # --------------------------------------------
                # Extract image time and nearest met data
                # --------------------------------------------

                image_time_utc = extract_datetime_from_filename(file)

                if image_time_utc is None:
                    print(f"[!] Could not extract datetime from: {file}")
                    print("    Skipping plot because meteorological matching requires image time.")
                    failed_count += 1
                    cleanup_loop_variables()
                    continue

                image_time_local = image_time_utc.astimezone(pacific)

                wspd, wdir, met_time = get_nearest_met_data(image_time_local)

                print(f"  image time local: {image_time_local.strftime('%Y-%m-%d %H:%M:%S %Z')}")
                print(f"  nearest met time: {met_time.strftime('%Y-%m-%d %H:%M:%S %Z')}")
                print(f"  wind: {wspd:.2f} m/s @ {wdir:.1f} deg")

                # --------------------------------------------
                # Read original SAR image
                # --------------------------------------------

                print("  reading original SAR raster...")

                with rasterio.open(original_path) as src_orig:
                    original = src_orig.read(1).astype(np.float32)
                    bounds = src_orig.bounds
                    extent = [bounds.left, bounds.right, bounds.bottom, bounds.top]
                    original_shape = src_orig.shape
                    original_crs = src_orig.crs

                print_memory("after reading original raster")

                # Metadata checks
                if original_shape != mask_shape:
                    print("[!] Warning: original image and mask have different shapes.")
                    print(f"    Original shape: {original_shape}")
                    print(f"    Mask shape:     {mask_shape}")

                if original_crs != crs:
                    print("[!] Warning: original image and mask have different CRS.")
                    print(f"    Original CRS: {original_crs}")
                    print(f"    Mask CRS:     {crs}")

                # --------------------------------------------
                # Stretch original SAR image for display
                # --------------------------------------------

                print("  stretching original raster...")

                vmin, vmax = np.nanpercentile(original, (2, 95))

                if vmax == vmin:
                    print("[!] Warning: image has no contrast after percentile stretch.")
                    normed = np.zeros_like(original, dtype=np.float32)
                else:
                    clipped = np.clip(original, vmin, vmax)
                    normed = ((clipped - vmin) / (vmax - vmin)).astype(np.float32)

                print_memory("after stretching original raster")

                # Delete original because normed is what gets plotted
                del original
                gc.collect()
                print_memory("after deleting original raster")

                # --------------------------------------------
                # Build lightweight plume overlay
                # --------------------------------------------

                print("  creating lightweight overlay...")

                # This is much smaller than a full RGBA float array.
                # It plots 1 where the mask is valid and hides everything else.
                overlay = np.ma.masked_where(
                    ~valid_mask,
                    np.ones(valid_mask.shape, dtype=np.uint8)
                )

                print_memory("after creating lightweight overlay")

                # --------------------------------------------
                # Reproject points to image CRS
                # --------------------------------------------

                points_gdf = points_gdf_orig.to_crs(crs)

                # --------------------------------------------
                # Create plot
                # --------------------------------------------

                print("  creating plot...")

                fig, axes = plt.subplots(
                    1,
                    2,
                    figsize=(9, 12),
                    constrained_layout=True
                )

                # Left panel: original SAR
                axes[0].imshow(
                    normed,
                    cmap="gray",
                    extent=extent
                )

                points_gdf.plot(
                    ax=axes[0],
                    marker="o",
                    color="red",
                    markersize=5,
                    label="Outflow points"
                )

                axes[0].set_title("Original SAR (2-95% Stretch)", fontsize=12)
                axes[0].axis("off")

                # Right panel: original SAR with plume mask
                axes[1].imshow(
                    normed,
                    cmap="gray",
                    extent=extent
                )

                axes[1].imshow(
                    overlay,
                    cmap="autumn",
                    alpha=0.6,
                    extent=extent
                )

                points_gdf.plot(
                    ax=axes[1],
                    marker="o",
                    color="red",
                    markersize=5
                )

                axes[1].set_title("Original SAR with Plume Mask", fontsize=12)
                axes[1].axis("off")

                # --------------------------------------------
                # Add wind annotation
                # --------------------------------------------

                wind_text = (
                    f"Wind: {wspd:.1f} m/s @ {wdir:.0f} deg\n"
                    f"Image Time: {image_time_local.strftime('%Y-%m-%d %H:%M')} Pacific\n"
                    f"Met Time: {met_time.strftime('%Y-%m-%d %H:%M')} Pacific"
                )

                fig.text(
                    0.5,
                    0.15,
                    wind_text,
                    ha="center",
                    fontsize=13,
                    fontweight="bold"
                )

                # --------------------------------------------
                # Add wind arrow
                # --------------------------------------------

                wind_angle_deg = (270 - wdir) % 360
                angle_rad = math.radians(wind_angle_deg)

                height = bounds.top - bounds.bottom
                width = bounds.right - bounds.left

                base_frac = 0.07
                scale = np.clip(wspd / 10, 0.5, 1.5)

                arrow_length = height * base_frac * scale

                dx = arrow_length * math.cos(angle_rad)
                dy = arrow_length * math.sin(angle_rad)

                x0 = bounds.right - 0.1 * width
                y0 = bounds.bottom + 0.05 * height

                axes[1].add_patch(
                    patches.FancyArrow(
                        x0,
                        y0,
                        dx,
                        dy,
                        width=arrow_length * 0.05,
                        head_width=arrow_length * 0.15,
                        head_length=arrow_length * 0.15,
                        length_includes_head=True,
                        transform=axes[1].transData,
                        color="lime",
                        alpha=0.9
                    )
                )

                print_memory("after creating plot")

                # --------------------------------------------
                # Save figure
                # --------------------------------------------

                print(f"  saving plot: {out_png}")

                plt.savefig(
                    out_png,
                    dpi=150,
                    bbox_inches="tight"
                )

                plt.close(fig)

                print(f"  -> saved plot: {out_png}")
                print_memory("after saving plot")

                # Delete plotting objects immediately
                del fig
                del axes
                del normed
                del overlay
                del points_gdf

                try:
                    del clipped
                except NameError:
                    pass

                gc.collect()
                print_memory("after deleting plotting objects")

            elif make_plots and plot_done:
                print(f"  -> plot already exists: {out_png}")

            # ====================================================
            # Final cleanup for this file
            # ====================================================

            try:
                del mask_arr
            except NameError:
                pass

            try:
                del valid_mask
            except NameError:
                pass

            cleanup_loop_variables()
            print_memory("after final cleanup")

        except Exception as e:
            print(f"[!] Failed while processing: {file}")
            print(f"    Error: {e}")

            failed_count += 1

            cleanup_loop_variables()
            print_memory("after failed-file cleanup")

            continue


# ============================================================
# Final summary
# ============================================================

print("\n--------------------------------------------------")
print("Done!")
print(f"Files encountered: {processed_count:,}")
print(f"Files skipped because outputs already existed: {skipped_count:,}")
print(f"Files failed: {failed_count:,}")
print_memory("at end")

Loading meteorological data...
Loaded 193,348 valid meteorological records.
Loading outflow point shapefile...
Loaded 2 outflow point features.

Starting processing...
  memory before loop: 0.55 GB

--------------------------------------------------
File #1
Processing: S1A_IW_GRDH_1SDV_20220101T015007_20220101T015036_041261_04E75E_B606_pre_JPL1.3_VVDR_cumulative_mask.tif
  memory at start of file: 0.55 GB
  reading mask raster...
  memory after reading mask: 0.55 GB
  valid mask pixels: 39,757 / 6,375,336 (0.6236%)
  creating binary mask for vectorizing...
  memory after creating binary mask: 0.56 GB
  vectorizing mask...
  memory before vectorizing: 0.56 GB
  vectorizing complete.
  raw polygons created: 55
  kept polygons: 55
  skipped small polygons: 0
  skipped empty polygons: 0
  memory after vectorizing: 0.56 GB
  memory after deleting binary mask: 0.56 GB
  creating GeoDataFrame...
  memory after creating GeoDataFrame: 0.56 GB
  writing GeoPackage: /Volumes/External/TJ/015_shape

/opt/anaconda3/envs/RS/lib/python3.9/site-packages/numpy/lib/nanfunctions.py:1384: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(


  memory after deleting original raster: 7.12 GB
  creating lightweight overlay...
  memory after creating lightweight overlay: 7.12 GB
  creating plot...
  memory after creating plot: 7.17 GB
  saving plot: /Volumes/External/TJ/050_viz/autoSAR_seg_output/S1A_IW_GRDH_1SDV_20230812T015018_20230812T015047_049836_05FE55_42EE_pre_overlay.png
  -> saved plot: /Volumes/External/TJ/050_viz/autoSAR_seg_output/S1A_IW_GRDH_1SDV_20230812T015018_20230812T015047_049836_05FE55_42EE_pre_overlay.png
  memory after saving plot: 7.18 GB
  memory after deleting plotting objects: 7.18 GB
  memory after final cleanup: 7.18 GB

--------------------------------------------------
File #147
Processing: S1A_IW_GRDH_1SDV_20230812T135320_20230812T135345_049843_05FE9D_EDFA_pre_JPL1.3_VVDR_cumulative_mask.tif
  memory at start of file: 7.18 GB
[!] Original image not found for: S1A_IW_GRDH_1SDV_20230812T135320_20230812T135345_049843_05FE9D_EDFA_pre_JPL1.3_VVDR_cumulative_mask.tif
    Expected path: /Volumes/External